# 04c · Cost (C4)

**Purpose.** Feature size, time to build features, time to train the best head and GPU need, per representation and dataset.

**Inputs**
- Feature builders from `src/features/`, both datasets; best heads from `04a`

**Outputs**
- `results/04c_cost.csv`
- `report/tables/` cost table (D = 7200 / 864 / 2304 for GWT, 770 / 132 / 352 for ES)

## Colab setup
Run this first on Google Colab (Runtime → Change runtime type → **GPU**). Before running, upload
**`results.zip`** (results of notebooks 01-03) to `/content` with the Files panel on the left. The cell clones the
repo, downloads the processed data from the shared Drive folder, unpacks the results, and moves into this
notebook's folder so the rest runs unchanged. On a laptop it does nothing.

In [1]:
import glob
import os
import shutil
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ
REPO = "/content/gas-repr-guide"
REPO_URL = "https://github.com/YasserAlbogami/gas-repr-guide.git"
DATA_URL = "https://drive.google.com/drive/folders/1rNIcZkHDFhH2dMVq8TgNWt_0IH2yw2_I"   # processed data (Yasser)
DATA_DIRS = ["gas_wind_tunnel", "everyday_scents"]

if IN_COLAB:
    # 1. code (src/). Tried in this order:
    #    a) clone the GitHub repo (works when the repo is public)
    #    b) clone with a GitHub token saved in Colab Secrets (key icon on the left) as GITHUB_TOKEN
    #    c) src.zip uploaded to /content (Files panel on the left)
    if not os.path.exists(f"{REPO}/src"):
        env = dict(os.environ, GIT_TERMINAL_PROMPT="0")           # fail instead of asking for a password
        ok = subprocess.run(["git", "clone", "-q", REPO_URL, REPO], env=env).returncode == 0
        if not ok:
            token = None
            try:
                from google.colab import userdata
                token = userdata.get("GITHUB_TOKEN")
            except Exception:
                pass
            if token:
                url = REPO_URL.replace("https://", f"https://{token}@")
                ok = subprocess.run(["git", "clone", "-q", url, REPO], env=env).returncode == 0
        if not ok and os.path.exists("/content/src.zip"):
            os.makedirs(f"{REPO}/notebooks/03_encoder", exist_ok=True)
            shutil.unpack_archive("/content/src.zip", REPO)
            ok = True
        if not ok:
            raise RuntimeError("Could not get the code: the repo is private. Make it public, or add GITHUB_TOKEN "
                               "in Colab Secrets, or upload src.zip to /content, then run this cell again.")
    elif os.path.exists(f"{REPO}/.git"):
        subprocess.run(["git", "-C", REPO, "pull", "-q"], check=False)
    os.makedirs(f"{REPO}/results", exist_ok=True)

    # 2. processed data (only downloaded once per Colab session)
    processed = f"{REPO}/data/processed"
    if not all(os.path.exists(f"{processed}/{d}/X_signals.npy") for d in DATA_DIRS):
        import gdown
        tmp = "/content/processed_download"
        gdown.download_folder(DATA_URL, output=tmp, quiet=False)
        for d in DATA_DIRS:
            found = glob.glob(f"{tmp}/**/{d}", recursive=True)
            assert found, f"folder {d} not found in the Drive download"
            shutil.copytree(found[0], f"{processed}/{d}", dirs_exist_ok=True)
    for d in DATA_DIRS:
        files = sorted(os.listdir(f"{processed}/{d}"))
        print(d, files)
        for f in ["X_signals.npy", "meta.csv", "info.json"]:
            assert f in files, f"{d}/{f} is missing"

    # 2b. results of the earlier notebooks (01, 02, 03 and the 04 ones already run), from results.zip
    #     uploaded to /content (Files panel on the left). The zip holds a results/ folder (+ report/tables/).
    if os.path.exists("/content/results.zip"):
        shutil.unpack_archive("/content/results.zip", REPO)
    have = sorted(os.path.basename(f) for f in glob.glob(f"{REPO}/results/*.csv"))
    print("results:", have)
    main = ["01a_gwt_raw", "01b_es_raw", "02b_gwt_handcrafted", "02c_es_handcrafted",
            "03a_gwt_encoder", "03b_es_encoder", "03c_end_to_end_reference"]
    missing = [n for n in main if f"{n}.csv" not in have]
    assert not missing, f"upload results.zip to /content first (missing: {missing})"

    # 3. run the rest of the notebook from its own folder, as on a laptop (so ../.. = project root)
    os.chdir(f"{REPO}/notebooks/04_comparisons")

    import torch
    print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available()
          else "NONE, switch on Runtime > Change runtime type > GPU")

In [2]:
import sys
sys.path.append("../..")          # project root, so that `import src` works

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path("../..")
import time

import torch
from src.cv import load_baseline_level, load_processed
from src.features.handcrafted import handcrafted_features
from src.features.raw import raw_features

TABLES = ROOT / "report/tables"
RESULTS = ROOT / "results"
MAIN = ["01a_gwt_raw", "01b_es_raw", "02b_gwt_handcrafted", "02c_es_handcrafted",
        "03a_gwt_encoder", "03b_es_encoder", "03c_end_to_end_reference"]
missing = [n for n in MAIN if not (RESULTS / f"{n}.csv").exists()]
assert not missing, f"run these notebooks first: {missing}"
scores = pd.concat([pd.read_csv(RESULTS / f"{n}.csv") for n in MAIN], ignore_index=True)
REPS = ["R1_raw", "R2_handcrafted", "R3_encoder"]
HEADS = ["logreg", "svm", "rf", "mlp"]

## C4: cost of each representation
- **feature size**: D
- **time to build features**: R1 and R2 are timed here on the full dataset; for R3 it is the per-fold
  encoder training + encoding time recorded by `run_cv` (`feat_seconds`)
- **time to train the best head**: mean `fit_seconds` (grid search included) of the best head
- **GPU**: whether the representation needs one

In [3]:
build_time = {}
for dataset in ["gwt", "es"]:
    X, meta, info = load_processed(dataset)
    levels = load_baseline_level(dataset)
    t0 = time.perf_counter(); raw_features(X, 100 if dataset == "gwt" else 70); build_time[(dataset, "R1_raw")] = time.perf_counter() - t0
    t0 = time.perf_counter(); handcrafted_features(X, levels, info["n_base"], info["exposure_end"]); build_time[(dataset, "R2_handcrafted")] = time.perf_counter() - t0

rows = []
main = scores[scores["task"] == "multi"]
for (dataset, rep), s in main[main["rep"].isin(REPS)].groupby(["dataset", "rep"]):
    head = s.groupby("head")["macro_f1"].mean().idxmax()
    rows.append({
        "dataset": dataset, "rep": rep, "D": int(s["n_features"].iloc[0]),
        "build_features_s": build_time.get((dataset, rep), s["feat_seconds"].mean()),
        "best_head": head,
        "train_best_head_s": s[s["head"] == head]["fit_seconds"].mean(),
        "gpu": "optional (trains on CPU)" if rep == "R3_encoder" else "no",
    })
cost = pd.DataFrame(rows).round(2)
cost.to_csv(TABLES / "C4_cost.csv", index=False)
print("GPU available on this machine:", torch.cuda.is_available())
cost

GPU available on this machine: False


,dataset,rep,D,build_features_s,best_head,train_best_head_s,gpu
0,es,R1_raw,770,0.00,rf,0.36,no
1,es,R2_handcrafted,132,0.07,rf,0.31,no
2,es,R3_encoder,352,0.88,logreg,0.25,optional (trains on CPU)
3,gwt,R1_raw,7200,0.03,rf,3.54,no
4,gwt,R2_handcrafted,864,3.31,rf,1.40,no
5,gwt,R3_encoder,2304,4.05,rf,7.56,optional (trains on CPU)


## Colab: save the outputs
Colab deletes everything when the session ends. This zips `results/` and `report/` into `outputs_<notebook>.zip`,
copies it to your Google Drive (`MyDrive/gas-repr-guide/`) and downloads it. On a laptop it does nothing.

In [4]:
if IN_COLAB:
    out_zip = shutil.make_archive(f"/content/outputs_04c_cost", "zip", REPO, "results")
    shutil.make_archive(f"/content/report_04c_cost", "zip", REPO, "report")
    files_out = [out_zip, f"/content/report_04c_cost.zip"]
    print(files_out)
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        out_dir = "/content/drive/MyDrive/gas-repr-guide"
        os.makedirs(out_dir, exist_ok=True)
        for f in files_out:
            shutil.copy(f, out_dir)
        print("copied to", out_dir)
    except Exception as e:
        print("Drive copy failed:", e)
    from google.colab import files
    for f in files_out:
        files.download(f)